In [1]:
# ── Cell 1: Imports & Configuration ──────────────────────────────────────────
import requests
import pandas as pd
import re
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

# ── Only these two variables need to change ───────────────────────────────────
TICKER     = "OKE"
USER_AGENT = "sge42@txstate.edu"   # Required by SEC fair-use policy
# ─────────────────────────────────────────────────────────────────────────────

TODAY       = datetime.today().strftime('%Y-%m-%d')
OUTPUT_FILE = f"{TICKER}_10K_Historical_{TODAY}.xlsx"

In [2]:
# ── Cell 2: Core Helper Functions ────────────────────────────────────────────

HEADERS     = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate",
               "Host": "data.sec.gov"}
HEADERS_SEC = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate",
               "Host": "www.sec.gov"}

def fetch_json(url, retries=3):
    h = HEADERS if "data.sec.gov" in url else HEADERS_SEC
    for attempt in range(retries):
        try:
            time.sleep(0.15)
            r = requests.get(url, headers=h, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                raise e

def to_millions(value, unit):
    if value is None:
        return None
    return round(value / 1_000_000, 2) if unit in ("USD", "shares") else value

def get_annual_values(facts_gaap, tag, fy_end_set):
    """Return {fy_end: value_in_millions} for a single XBRL tag, 10-K only."""
    if tag not in facts_gaap:
        return {}
    units_dict = facts_gaap[tag].get("units", {})
    unit_key   = next((k for k in units_dict if k in ("USD", "shares", "pure")), None)
    if unit_key is None:
        return {}
    annual = [e for e in units_dict[unit_key]
              if e.get("form") in ("10-K", "10-K/A")]
    fy_map = {}
    for e in annual:
        end   = e.get("end")
        filed = e.get("filed", "")
        if end not in fy_end_set:
            continue
        if end not in fy_map or filed > fy_map[end]["filed"]:
            fy_map[end] = {"val": e.get("val"), "unit": unit_key, "filed": filed}
    return {end: to_millions(info["val"], info["unit"])
            for end, info in fy_map.items()}

In [3]:
# ── Cell 3: CIK Lookup & 10-K Filing Identification ──────────────────────────

tickers_data = fetch_json("https://www.sec.gov/files/company_tickers.json")
ticker_upper = TICKER.upper()
cik_raw = company_name = None

for entry in tickers_data.values():
    if entry["ticker"].upper() == ticker_upper:
        cik_raw      = entry["cik_str"]
        company_name = entry["title"]
        break

if cik_raw is None:
    raise ValueError(f"Ticker '{TICKER}' not found in SEC ticker list")

CIK = str(cik_raw).zfill(10)
print(f"\u2705 Resolved {TICKER} \u2192 {company_name} (CIK: {CIK})")

submissions = fetch_json(f"https://data.sec.gov/submissions/CIK{CIK}.json")
filings     = submissions.get("filings", {}).get("recent", {})

tenk_filings = [
    {"accessionNumber": filings["accessionNumber"][i],
     "filingDate":      filings["filingDate"][i],
     "reportDate":      filings["reportDate"][i],
     "primaryDocument": filings["primaryDocument"][i]}
    for i, form in enumerate(filings.get("form", []))
    if form in ("10-K", "10-K/A")
]

tenk_filings.sort(key=lambda x: x["filingDate"], reverse=True)
tenk_filings = tenk_filings[:5]
tenk_filings.sort(key=lambda x: x["reportDate"])    # oldest → newest

FY_ENDS   = [f["reportDate"] for f in tenk_filings]
FY_LABELS = [f"FY{f['reportDate'][:4]}" for f in tenk_filings]

MOST_RECENT_YEAR = int(FY_ENDS[-1][:4])
PROJ_LABELS      = [f"FY{MOST_RECENT_YEAR + i}E" for i in range(1, 4)]

print(f"\u2705 Found {len(tenk_filings)} 10-K filings : {', '.join(FY_LABELS)}")
print(f"\u2705 Forecast columns              : {', '.join(PROJ_LABELS)}")

✅ Resolved OKE → ONEOK INC /NEW/ (CIK: 0001039684)
✅ Found 5 10-K filings : FY2021, FY2022, FY2023, FY2024, FY2025
✅ Forecast columns              : FY2026E, FY2027E, FY2028E


In [4]:
# ── Cell 4: XBRL Company Facts Download ──────────────────────────────────────
# Kept for scale calibration and the Raw Facts audit sheet.

facts_data = fetch_json(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json")
facts_gaap = facts_data.get("facts", {}).get("us-gaap", {})
print(f"\u2705 Pulled XBRL company facts  ({len(facts_gaap)} us-gaap concepts)")

✅ Pulled XBRL company facts  (565 us-gaap concepts)


In [5]:
# ── Cell 5: As-Reported Statement Harvester ──────────────────────────────────
#
# Pulls each statement exactly as rendered in the SEC XBRL Viewer (R-files)
# for each 10-K filing, preserving:
#   • the exact filing order of every disclosed line item
#   • the indentation hierarchy used by the company's accountants
#   • all XBRL-tagged values (converted to $MM)
#
# For each filing:
#   1. MetaLinks.json  →  map statement type to R-file number
#   2. R{n}.htm        →  parse labels in order + indent level + raw value str
#   3. Scale detection →  caption text  →  calibrate vs. XBRL company facts
#   4. Merge years     →  most-recent year sets row order; gaps = None
#
# Outputs: df_is, df_bs, df_cfs   (same interface as prior cells)
#          is_levels, bs_levels, cfs_levels  ({label: int indent in em})

# ── Statement identification ──────────────────────────────────────────────────
_STMT_KEYS = {
    "IS":  ["statement of income", "statements of income",
            "statement of operations", "statements of operations",
            "statement of earnings",   "statements of earnings"],
    "BS":  ["balance sheet", "statements of condition",
            "financial position",      "consolidated balance"],
    "CFS": ["cash flow", "statements of cash"],
}
_SKIP_NAMES = {
    "parenthetical", "supplemental", "note ", "other comprehensive",
    "geographic",    "quarterly",    "segment",
}

def _metalinks(cik, acc):
    """
    Fetch MetaLinks.json and return the sub-dict that contains 'report'.
    SEC filing structure (v2.2+): report is nested under instance → filing_key.
    Fall back to top-level for older format.
    """
    a = acc.replace("-", "")
    try:
        ml = fetch_json(
            f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{a}/MetaLinks.json")
    except Exception:
        return None
    if ml is None:
        return None
    # Newer format: {"instance": {"filing.htm": {"report": {...}, ...}}}
    instance = ml.get("instance", {})
    for key, val in instance.items():
        if isinstance(val, dict) and "report" in val:
            return val          # return the inner dict that has "report"
    # Older format: "report" at top level
    if "report" in ml:
        return ml
    return ml                   # return as-is; _find_r will handle gracefully

def _find_r(metalinks, stype):
    """
    Return best R-file number for a statement type, or None.
    Searches longName + shortName + menuCat; broad single-word fallback.
    """
    if not metalinks:
        return None

    _PRECISE = {
        "IS":  ["statement of income", "statements of income",
                "statement of operations", "statements of operations",
                "statement of earnings",   "statements of earnings",
                "statements of comprehensive income"],
        "BS":  ["balance sheet", "statements of condition",
                "financial position", "consolidated balance"],
        "CFS": ["cash flow", "statements of cash", "cash flows"],
    }
    _BROAD = {
        "IS":  ["income", "operations", "earnings"],
        "BS":  ["balance"],
        "CFS": ["cash"],
    }

    precise = _PRECISE[stype]
    broad   = _BROAD[stype]
    reports = metalinks.get("report", {})
    best    = (0, None)

    for rk, meta in reports.items():
        ln = meta.get("longName",  "").lower()
        sn = meta.get("shortName", "").lower()
        mc = meta.get("menuCat",   "").lower()
        nm = f"{ln} {sn} {mc}"

        if any(w in nm for w in _SKIP_NAMES):
            continue

        score  = sum(2 for k in precise if k in nm)
        score += sum(1 for k in broad   if k in nm)
        if meta.get("groupType", "").lower() == "statement":
            score += 0.5

        if score > best[0]:
            try:
                best = (score, int(rk[1:]))
            except (ValueError, IndexError):
                pass

    return best[1]


def _r_html(cik, acc, rn):
    a   = acc.replace("-", "")
    url = (f"https://www.sec.gov/Archives/edgar/data/"
           f"{int(cik)}/{a}/R{rn}.htm")
    time.sleep(0.2)
    try:
        r = requests.get(url, headers=HEADERS_SEC, timeout=30)
        return r.text if r.status_code == 200 else None
    except Exception:
        return None

def _scale_from_text(html):
    """$MM multiplier inferred from HTML caption text; None if not found."""
    t = (html or "").lower()
    if re.search(r'in\s+billions',  t): return 1_000.0
    if re.search(r'in\s+millions',  t): return 1.0
    if re.search(r'in\s+thousands', t): return 0.001
    if re.search(r'\$000s?\b',     t): return 0.001
    return None

# Calibration tags used when caption scale is absent
_CAL_TAGS = {
    "IS":  ["Revenues",
            "RevenueFromContractWithCustomerExcludingAssessedTax",
            "InterestAndDividendIncomeOperating"],
    "BS":  ["Assets"],
    "CFS": ["NetCashProvidedByUsedInOperatingActivities"],
}

def _calibrate_scale(parsed_rows, facts_gaap, fy_end, stype):
    """
    Infer scale by comparing the first large R-file numeric value to
    the equivalent value from XBRL company facts (already in $MM).
    """
    fy_set = {fy_end}
    cf_mm  = None
    for tag in _CAL_TAGS.get(stype, []):
        m = get_annual_values(facts_gaap, tag, fy_set)
        if m.get(fy_end) is not None:
            cf_mm = abs(m[fy_end])
            break
    if not cf_mm or cf_mm < 0.01:
        return 1.0
    for _, _, vs in parsed_rows:
        s = vs.replace(",", "").replace("$", "").strip()
        if s.startswith("(") and s.endswith(")"):
            s = s[1:-1]
        try:
            raw = abs(float(s))
        except (ValueError, TypeError):
            continue
        if raw < 1:
            continue
        ratio = cf_mm / raw
        for scale in (1.0, 0.001, 1_000.0, 1e-6, 1e6):
            if 0.70 < ratio / scale < 1.43:
                return scale
    return 1.0

def _parse_r(html, fy_end):
    """
    Parse R-file HTML into ordered list of (label, indent_em, raw_val_str).

    SEC R-files have wildly inconsistent header structures across companies
    (1-row, 2-row, colspan=2 label merges, etc.).  Trying to parse headers
    is fragile.  Instead we auto-detect the first data column:

    Invariant: for every 10-K the FIRST numeric column after the label
    column is the fiscal year being reported.  Companies always list the
    current year first.  We find that column by scanning the first ~12 rows
    for whichever column index consistently contains numeric values, then
    skip any non-data header/date rows at the top.
    """
    if not html:
        return []
    soup  = BeautifulSoup(html, "html.parser")
    table = soup.find("table")
    if not table:
        return []
    rows = table.find_all("tr")
    if len(rows) < 2:
        return []

    def _is_num(text):
        """True if text looks like a reported financial figure."""
        s = (text.replace(",", "").replace("$", "")
                 .replace("(", "").replace(")", "")
                 .replace("\u2014", "").replace("\u2013", "")
                 .replace("-", "").strip())
        return s.isdigit() and len(s) > 0

    # ── Step 1: find the leftmost column that holds numeric data ─────────────
    # Scan rows 1-12 (skip row 0 which is often a merged title row)
    col_counts = {}
    for row in rows[1:13]:
        cells = row.find_all(["td", "th"])
        for ci in range(1, len(cells)):          # col 0 is always the label
            if _is_num(cells[ci].get_text(strip=True)):
                col_counts[ci] = col_counts.get(ci, 0) + 1

    # Choose the leftmost column that appears numeric in at least 2 rows
    # (guards against stray numbers in header/footnote cells)
    candidates = sorted(ci for ci, cnt in col_counts.items() if cnt >= 2)
    col_idx = candidates[0] if candidates else 1

    # ── Step 2: skip header/date rows at the top ─────────────────────────────
    # A row is a data row if at least one cell at col_idx or beyond is numeric.
    data_start = 1
    for ri, row in enumerate(rows[1:6], 1):
        cells = row.find_all(["td", "th"])
        if len(cells) > col_idx and _is_num(cells[col_idx].get_text(strip=True)):
            data_start = ri
            break

    # ── Step 3: extract labels + values ──────────────────────────────────────
    result = []
    for row in rows[data_start:]:
        cells = row.find_all(["td", "th"])
        if len(cells) <= col_idx:
            continue
        lc    = cells[0]
        label = re.sub(r'\s+', ' ',
                       re.sub(r'\[\d+\]', '',
                              lc.get_text(" ", strip=True))).strip()
        if not label:
            continue
        sm     = re.search(r'padding-left:\s*([\d.]+)em', lc.get("style", ""))
        indent = round(float(sm.group(1))) if sm else 0
        val_str = cells[col_idx].get_text(strip=True)
        result.append((label, indent, val_str))
    return result


def _parse_val(vs, scale):
    """Convert R-file value string → float in $MM, or None."""
    if not vs or vs in ("\u2014", "\u2013", "-", "", "N/A"):
        return None
    s   = vs.replace(",", "").replace("$", "").strip()
    neg = s.startswith("(") and s.endswith(")")
    if neg:
        s = s[1:-1]
    try:
        return round(float(s) * scale * (-1 if neg else 1), 3)
    except (ValueError, TypeError):
        return None

# ── Harvest loop ──────────────────────────────────────────────────────────────
_raw = {"IS": {}, "BS": {}, "CFS": {}}
print("Fetching as-reported statements from SEC XBRL Viewer…\n")

for filing in tenk_filings:
    acc    = filing["accessionNumber"]
    fy_e   = filing["reportDate"]
    fy_lbl = f"FY{fy_e[:4]}"
    ml     = _metalinks(CIK, acc)

    if not ml:
        print(f"  \u26a0  {fy_lbl}: MetaLinks not found — skipping")
        for st in _raw:
            _raw[st][fy_lbl] = []
        continue

    for st in ("IS", "BS", "CFS"):
        rn = _find_r(ml, st)
        if rn is None:
            print(f"  \u26a0  {fy_lbl} {st}: R-file not identified in MetaLinks")
            _raw[st][fy_lbl] = []
            continue
        html  = _r_html(CIK, acc, rn)
        rows  = _parse_r(html, fy_e)
        scale = _scale_from_text(html) or _calibrate_scale(rows, facts_gaap, fy_e, st)
        final = [(lbl, ind, _parse_val(vs, scale)) for lbl, ind, vs in rows]
        _raw[st][fy_lbl] = final
        print(f"  \u2705 {fy_lbl} {st} \u2192 R{rn:02d}  "
              f"({len(final)} rows, scale \u00d7{scale:.4g})")

# ── Build DataFrames preserving filing order ──────────────────────────────────
def _build(raw_by_fy, fy_labels):
    """
    Merge per-year rows into a wide DataFrame.
    Most-recent year defines row order; older years append new labels at end.
    Returns (df, {label: indent_em}).
    """
    order, levels, seen = [], {}, set()
    for fy in reversed(fy_labels):          # most-recent sets ordering
        for lbl, ind, _ in raw_by_fy.get(fy, []):
            if lbl not in seen:
                order.append(lbl)
                levels[lbl] = ind
                seen.add(lbl)
    data = {}
    for lbl in order:
        row = []
        for fy in fy_labels:
            ym = {r[0]: r[2] for r in raw_by_fy.get(fy, [])}
            row.append(ym.get(lbl))
        data[lbl] = row
    df = pd.DataFrame(data, index=fy_labels).T
    df.index.name = "Line Item"
    return df, levels

df_is,  is_levels  = _build(_raw["IS"],  FY_LABELS)
df_bs,  bs_levels  = _build(_raw["BS"],  FY_LABELS)
df_cfs, cfs_levels = _build(_raw["CFS"], FY_LABELS)

print(f"\n\u2705 Income Statement  : {len(df_is)} line items")
print(f"\u2705 Balance Sheet     : {len(df_bs)} line items")
print(f"\u2705 Cash Flow         : {len(df_cfs)} line items")

Fetching as-reported statements from SEC XBRL Viewer…

  ✅ FY2021 IS → R03  (29 rows, scale ×0.001)
  ✅ FY2021 BS → R07  (42 rows, scale ×0.001)
  ✅ FY2021 CFS → R09  (34 rows, scale ×0.001)
  ✅ FY2022 IS → R03  (29 rows, scale ×0.001)


KeyboardInterrupt: 

In [ ]:
# ── Cell 6: Excel Workbook Builder ───────────────────────────────────────────
#
# Column layout per statement sheet
#   A        : margin (width 3)
#   B        : row labels with visual indentation (width 42)
#   C … C+n-1: historical FY actuals (blue)
#   C+n …   : forecast cols (light-blue, user-editable)
#
# Rows are styled by type:
#   section_header  →  LIGHT_GRAY fill, bold — marks filing sections
#   subtotal/total  →  SUBTOTAL_C fill, bold, thin top border
#   normal          →  alternating white / ALT_ROW

# ── Style constants ───────────────────────────────────────────────────────────
NAVY       = "1F3864"
MED_GRAY   = "BFBFBF"
LIGHT_GRAY = "F2F2F2"
SUBTOTAL_C = "E8E8E8"
ALT_ROW    = "FAFAFA"
PROJ_FILL  = "EBF3FB"
WACC_INPUT = "FFF2CC"
WACC_RSLT  = "D9EAD3"

THIN_TOP = Border(top=Side(style="thin"))

FMT_DOLLAR = '$#,##0.0;($#,##0.0);"-"'
FMT_PCT    = '0.0%;(0.0%);"-"'
FMT_EPS    = '$#,##0.00;($#,##0.00);"-"'
FMT_SHARES = '#,##0.0;(#,##0.0);"-"'
FMT_2DP    = '#,##0.00'

N_FY           = len(FY_LABELS)
N_PROJ         = len(PROJ_LABELS)
DATA_START_COL = 3
DATA_END_COL   = DATA_START_COL + N_FY - 1
PROJ_START_COL = DATA_END_COL + 1

def col(n): return get_column_letter(n)

# ── Dynamic label classifiers ─────────────────────────────────────────────────
_SUBTOTAL_RE = re.compile(
    r'^\s*total\b|\btotal\s|^\s*net\s+income|^\s*net\s+loss'
    r'|^\s*gross\s+profit|^\s*income\s+before\b|^\s*earnings\s+before\b'
    r'|^\s*net\s+cash\s+(provided|used)|^\s*operating\s+income'
    r'|^\s*net\s+interest\s+income|^\s*net\s+revenue|^\s*total\s+net',
    re.I)

def _auto_subtotals(df):
    return {lbl for lbl in df.index if _SUBTOTAL_RE.search(lbl)}

def _auto_section_hdrs(df, levels):
    """
    Section headers: indent=0 rows with all-None values (a grouping label),
    or any label ending with ':'.
    """
    result = set()
    for lbl in df.index:
        if lbl.strip().endswith(":"):
            result.add(lbl); continue
        if levels.get(lbl, 0) == 0:
            vals = [v for v in df.loc[lbl]
                    if v is not None and not (isinstance(v, float) and pd.isna(v))]
            if not vals:
                result.add(lbl)
    return result

def _num_fmt(label):
    l = label.lower()
    if ("per share" in l or "per common share" in l) and "weighted" not in l:
        return FMT_EPS
    if "weighted average" in l and ("share" in l or "unit" in l):
        return FMT_SHARES
    return FMT_DOLLAR

def _row(row_map, *candidates):
    """Return first matching row number from candidates, or None."""
    for c in candidates:
        v = row_map.get(c)
        if v is not None:
            return v
    return None

# ── Sheet helpers ─────────────────────────────────────────────────────────────
def set_header(ws, company, stmt_name):
    last_col = col(PROJ_START_COL + N_PROJ - 1)
    ws.row_dimensions[1].height = 8
    ws.merge_cells(f"B2:{last_col}2")
    c = ws["B2"]
    c.value     = f"{company}  |  {stmt_name}"
    c.font      = Font(bold=True, size=14, color="FFFFFF", name="Calibri")
    c.fill      = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[2].height = 24
    ws["B3"].value = "All figures in $MM unless noted"
    ws["B3"].font  = Font(italic=True, size=9, color="808080", name="Calibri")
    ws.row_dimensions[3].height = 14
    hdr_fill = PatternFill("solid", fgColor=MED_GRAY)
    hdr_font = Font(bold=True, size=10, name="Calibri")
    for i, fy in enumerate(FY_LABELS):
        c = ws.cell(row=4, column=DATA_START_COL+i, value=fy)
        c.font = hdr_font; c.fill = hdr_fill
        c.alignment = Alignment(horizontal="center")
    proj_fill = PatternFill("solid", fgColor="D9D9D9")
    for i, pl in enumerate(PROJ_LABELS):
        c = ws.cell(row=4, column=PROJ_START_COL+i, value=pl)
        c.font = Font(bold=True, size=10, name="Calibri", color="595959")
        c.fill = proj_fill; c.alignment = Alignment(horizontal="center")

def set_col_widths(ws):
    ws.column_dimensions["A"].width = 3
    ws.column_dimensions["B"].width = 42       # wider for indented labels
    for i in range(N_FY):
        ws.column_dimensions[col(DATA_START_COL+i)].width = 13
    for i in range(N_PROJ):
        ws.column_dimensions[col(PROJ_START_COL+i)].width = 13

def write_data_rows(ws, df, start_row, subtotals,
                    section_headers=None, levels=None):
    """
    Write DataFrame rows to worksheet.
    section_headers : set of label strings → styled as section dividers
    levels          : {label: indent_em} → adds leading spaces to label text
    Returns (next_row, {label: sheet_row_number}).
    """
    sh    = section_headers or set()
    lvls  = levels or {}
    r     = start_row
    alt   = False
    rmap  = {}

    for label in df.index:
        is_hdr = label in sh
        is_sub = (label in subtotals) and not is_hdr

        if is_hdr:
            # Section header styling: no value, dark label, LIGHT_GRAY fill
            indent_txt = "  " * lvls.get(label, 0) + label
            lc = ws.cell(row=r, column=2, value=indent_txt)
            lc.font = Font(bold=True, size=10, name="Calibri")
            for cn in range(2, PROJ_START_COL + N_PROJ + 1):
                ws.cell(row=r, column=cn).fill = PatternFill("solid", fgColor=LIGHT_GRAY)
            ws.row_dimensions[r].height = 16
            rmap[label] = r
            r += 1; alt = False
            continue

        bg    = SUBTOTAL_C if is_sub else (ALT_ROW if alt else "FFFFFF")
        rfill = PatternFill("solid", fgColor=bg)
        nfmt  = _num_fmt(label)

        indent_txt = "  " * lvls.get(label, 0) + label
        lc = ws.cell(row=r, column=2, value=indent_txt)
        lc.font = Font(bold=is_sub, size=10, name="Calibri")
        lc.fill = rfill

        for i, fy in enumerate(FY_LABELS):
            v  = df.loc[label, fy]
            dc = ws.cell(row=r, column=DATA_START_COL+i,
                         value=(float(v) if pd.notna(v) and v is not None else None))
            dc.number_format = nfmt
            dc.font      = Font(color="0000FF", name="Calibri", size=10, bold=is_sub)
            dc.fill      = rfill
            dc.alignment = Alignment(horizontal="right")
            if is_sub: dc.border = THIN_TOP

        pfill = PatternFill("solid", fgColor=PROJ_FILL)
        for i in range(N_PROJ):
            pc = ws.cell(row=r, column=PROJ_START_COL+i)
            pc.number_format = nfmt
            pc.font  = Font(color="000000", name="Calibri", size=10)
            pc.fill  = pfill
            pc.alignment = Alignment(horizontal="right")

        rmap[label] = r
        ws.row_dimensions[r].height = 15
        r += 1; alt = not alt

    return r, rmap

def write_section_hdr(ws, r, label, fg=LIGHT_GRAY):
    ws.cell(row=r, column=2, value=label).font = Font(bold=True, size=10, name="Calibri")
    for cn in range(2, PROJ_START_COL + N_PROJ + 1):
        ws.cell(row=r, column=cn).fill = PatternFill("solid", fgColor=fg)
    ws.row_dimensions[r].height = 16

def write_formula_row(ws, r, label, fn, num_fmt=FMT_PCT):
    ws.cell(row=r, column=2, value=label).font = Font(size=10, name="Calibri")
    for i in range(N_FY):
        dc = ws.cell(row=r, column=DATA_START_COL+i, value=fn(r, DATA_START_COL+i))
        dc.number_format = num_fmt
        dc.font      = Font(color="000000", name="Calibri", size=10)
        dc.alignment = Alignment(horizontal="right")
    ws.row_dimensions[r].height = 15

def write_stats_block(ws, start_r, row_map, stat_labels, fmt_map=None):
    write_section_hdr(ws, start_r, "SUMMARY STATISTICS", fg="DEE6EF")
    hdr_r = start_r + 1
    for cn, txt in [(2, "Line Item"), (3, "5-Yr CAGR"),
                    (4, f"5-Yr Avg  ({FY_LABELS[0]}\u2013{FY_LABELS[-1]})")]:
        c = ws.cell(row=hdr_r, column=cn, value=txt)
        c.font = Font(bold=True, size=9, name="Calibri", color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="2E5F8A")
        c.alignment = Alignment(horizontal="center")
    ws.row_dimensions[hdr_r].height = 14

    c_s = col(DATA_START_COL); c_e = col(DATA_END_COL)
    n_p = N_FY - 1 if N_FY > 1 else 1
    r   = hdr_r + 1; alt = False

    for label in stat_labels:
        dr = row_map.get(label)
        if dr is None: continue
        fill    = PatternFill("solid", fgColor=ALT_ROW if alt else "FFFFFF")
        num_fmt = (fmt_map or {}).get(label, FMT_DOLLAR)
        lc = ws.cell(row=r, column=2, value=label)
        lc.font = Font(size=9, name="Calibri"); lc.fill = fill
        cc = ws.cell(row=r, column=3,
                     value=f"=IFERROR(({c_e}{dr}/{c_s}{dr})^(1/{n_p})-1,\"-\")")
        cc.number_format = FMT_PCT; cc.font = Font(bold=True, size=9, name="Calibri")
        cc.fill = fill; cc.alignment = Alignment(horizontal="center")
        ac = ws.cell(row=r, column=4,
                     value=f"=IFERROR(AVERAGE({c_s}{dr}:{c_e}{dr}),\"-\")")
        ac.number_format = num_fmt; ac.font = Font(bold=True, size=9, name="Calibri")
        ac.fill = fill; ac.alignment = Alignment(horizontal="right")
        ws.row_dimensions[r].height = 14
        r += 1; alt = not alt
    ws.column_dimensions[col(3)].width = max(ws.column_dimensions[col(3)].width or 0, 12)
    ws.column_dimensions[col(4)].width = max(ws.column_dimensions[col(4)].width or 0, 18)
    return r

# ── Build workbook ────────────────────────────────────────────────────────────
wb = Workbook()
wb.remove(wb.active)

# ── COVER SHEET ───────────────────────────────────────────────────────────────
cover = wb.create_sheet("Cover")
cover.column_dimensions["A"].width = 4
cover.column_dimensions["B"].width = 30
cover.column_dimensions["C"].width = 44
cover.merge_cells("B2:D2")
t = cover["B2"]
t.value     = f"{company_name}  |  SEC 10-K Financial Model"
t.font      = Font(bold=True, size=16, color="FFFFFF", name="Calibri")
t.fill      = PatternFill("solid", fgColor=NAVY)
t.alignment = Alignment(horizontal="left", vertical="center")
cover.row_dimensions[2].height = 30
for i, (k, v) in enumerate([
    ("Company",       company_name),
    ("Ticker",        TICKER.upper()),
    ("CIK",           CIK),
    ("Date Pulled",   TODAY),
    ("Fiscal Years",  ", ".join(FY_LABELS)),
    ("Forecast Cols", ", ".join(PROJ_LABELS)),
    ("Units",         "All values in $MM unless per-share"),
    ("Data Source",   "SEC EDGAR XBRL Viewer R-files (as reported)"),
], start=4):
    cover.cell(row=i, column=2, value=k).font = Font(bold=True, name="Calibri", size=10)
    cover.cell(row=i, column=3, value=v).font = Font(name="Calibri", size=10)
cover.cell(row=13, column=2, value="Contents").font = Font(bold=True, size=11, name="Calibri")
for i, sname in enumerate(["Income Statement", "Balance Sheet",
                            "Cash Flow Statement", "WACC", "As-Reported Labels"], start=14):
    c = cover.cell(row=i, column=2, value=sname)
    c.hyperlink = f"#{sname}!A1"
    c.font = Font(color="0563C1", underline="single", name="Calibri", size=10)

# ── INCOME STATEMENT SHEET ────────────────────────────────────────────────────
ws_is = wb.create_sheet("Income Statement")
set_col_widths(ws_is); set_header(ws_is, company_name, "Income Statement")

sub_is  = _auto_subtotals(df_is)
shdr_is = _auto_section_hdrs(df_is, is_levels)
next_r, is_map = write_data_rows(ws_is, df_is, start_row=5,
                                 subtotals=sub_is, section_headers=shdr_is,
                                 levels=is_levels)

# Margins block — labels may differ by company; uses fuzzy lookup
next_r += 1; write_section_hdr(ws_is, next_r, "MARGINS & RATIOS"); next_r += 1
rv = _row(is_map,
    "Total net interest and noninterest income", "Total revenues",
    "Net revenue", "Revenues", "Revenue",
    "Total interest and noninterest income")
gp = _row(is_map, "Gross profit", "Gross Profit")
oi = _row(is_map, "Operating income", "Income from operations",
          "Income before income taxes", "Operating Income")
ni = _row(is_map,
    "Net income attributable to Popular, Inc.", "Net income",
    "Net Income", "Net income attributable to common shareholders")
tx = _row(is_map, "Income tax expense", "Income taxes",
          "Provision for income taxes", "Income Tax Expense")
pt = _row(is_map, "Income before income taxes",
          "Pretax income", "Pretax Income")

def mfn(nr, dr):
    if nr and dr:
        return lambda r, c: f"=IFERROR({col(c)}{nr}/{col(c)}{dr},\"-\")"
    return lambda r, c: None

for lbl, fn in [
    ("Gross Margin %",          mfn(gp, rv)),
    ("Operating Margin %",      mfn(oi, rv)),
    ("Net Margin %",            mfn(ni, rv)),
    ("Effective Tax Rate %",    mfn(tx, pt)),
]:
    write_formula_row(ws_is, next_r, lbl, fn, FMT_PCT); next_r += 1

if rv:
    ws_is.cell(row=next_r, column=2, value="YoY Revenue Growth %").font = Font(size=10, name="Calibri")
    for i in range(N_FY):
        if i == 0: continue
        dc = ws_is.cell(row=next_r, column=DATA_START_COL+i,
                        value=f"=IFERROR({col(DATA_START_COL+i)}{rv}"
                              f"/{col(DATA_START_COL+i-1)}{rv}-1,\"-\")")
        dc.number_format = FMT_PCT
        dc.font = Font(color="000000", name="Calibri", size=10)
        dc.alignment = Alignment(horizontal="right")
    ws_is.row_dimensions[next_r].height = 15; next_r += 1

next_r += 1
stat_is = [l for l in [
    _row(is_map, "Total revenues", "Net revenue", "Revenues", "Revenue",
         "Total net interest and noninterest income"),
    _row(is_map, "Net income attributable to Popular, Inc.", "Net income", "Net Income"),
] if l is not None]
# Also find EPS diluted label by pattern
eps_lbl = next((l for l in is_map if "diluted" in l.lower() and "per share" in l.lower()), None)
if eps_lbl: stat_is.append(eps_lbl)
stat_labels_is = [lbl for lbl, rn in is_map.items()
                  if rn in stat_is or lbl in {eps_lbl}]
next_r = write_stats_block(ws_is, next_r, is_map,
    stat_labels_is,
    {eps_lbl: FMT_EPS} if eps_lbl else {})

# ── BALANCE SHEET SHEET ───────────────────────────────────────────────────────
ws_bs = wb.create_sheet("Balance Sheet")
set_col_widths(ws_bs); set_header(ws_bs, company_name, "Balance Sheet")

sub_bs  = _auto_subtotals(df_bs)
shdr_bs = _auto_section_hdrs(df_bs, bs_levels)
next_r_bs, bs_map = write_data_rows(ws_bs, df_bs, start_row=5,
                                    subtotals=sub_bs, section_headers=shdr_bs,
                                    levels=bs_levels)

next_r_bs += 1; write_section_hdr(ws_bs, next_r_bs, "KEY RATIOS"); next_r_bs += 1
ca  = _row(bs_map, "Total current assets",  "Total Current Assets",  "Total assets")
cl  = _row(bs_map, "Total current liabilities", "Total Current Liabilities")
te  = _row(bs_map, "Total stockholders' equity", "Total equity", "Total Equity",
           "Total shareholders' equity")
ltd = _row(bs_map, "Long-term debt", "Long-term borrowings", "Long-Term Debt")
csh = _row(bs_map, "Cash and cash equivalents", "Cash & Equivalents",
           "Cash and due from banks")
ta  = _row(bs_map, "Total assets", "Total Assets")

for lbl, fn, fmt in [
    ("Current Ratio",
     (lambda r, c: f"=IFERROR({col(c)}{ca}/{col(c)}{cl},\"-\")") if ca and cl else (lambda r,c: None),
     FMT_2DP),
    ("Total Debt / Total Assets",
     (lambda r, c: f"=IFERROR({col(c)}{ltd}/{col(c)}{ta},\"-\")") if ltd and ta else (lambda r,c: None),
     FMT_2DP),
    ("Net Debt ($MM)",
     (lambda r, c: f"=IFERROR({col(c)}{ltd}-{col(c)}{csh},\"-\")") if ltd and csh else (lambda r,c: None),
     FMT_DOLLAR),
]:
    write_formula_row(ws_bs, next_r_bs, lbl, fn, fmt); next_r_bs += 1

next_r_bs += 1
bs_stat_lbls = [lbl for lbl in bs_map
                if _row(bs_map, lbl) and
                any(k in lbl.lower() for k in ("total assets", "total liabilities",
                                                "total equity", "total deposits"))]
next_r_bs = write_stats_block(ws_bs, next_r_bs, bs_map, bs_stat_lbls)

# ── CASH FLOW STATEMENT SHEET ─────────────────────────────────────────────────
ws_cfs = wb.create_sheet("Cash Flow Statement")
set_col_widths(ws_cfs); set_header(ws_cfs, company_name, "Cash Flow Statement")

sub_cfs  = _auto_subtotals(df_cfs)
shdr_cfs = _auto_section_hdrs(df_cfs, cfs_levels)
next_r_cfs, cfs_map = write_data_rows(ws_cfs, df_cfs, start_row=5,
                                      subtotals=sub_cfs, section_headers=shdr_cfs,
                                      levels=cfs_levels)

next_r_cfs += 1; write_section_hdr(ws_cfs, next_r_cfs, "KEY METRICS"); next_r_cfs += 1
ocf_r = _row(cfs_map,
    "Net cash provided by operating activities",
    "Net cash provided by (used in) operating activities",
    "Net Cash Provided by Operating Activities")
cpx_r = _row(cfs_map,
    "Purchases of premises and equipment",
    "Capital expenditures", "Purchase of property and equipment",
    "Payments to acquire property plant and equipment")
ni_c  = _row(cfs_map, "Net income", "Net income attributable to Popular, Inc.",
             "Net Income (CFS)")

for lbl, fn, fmt in [
    ("CapEx % of Total Assets",
     (lambda r, c: f"=IFERROR(ABS({col(c)}{cpx_r})/"
                   f"'Balance Sheet'!{col(c)}{ta},\"-\")") if cpx_r and ta else (lambda r,c: None),
     FMT_PCT),
    ("OCF / Net Income",
     (lambda r, c: f"=IFERROR({col(c)}{ocf_r}/{col(c)}{ni_c},\"-\")") if ocf_r and ni_c else (lambda r,c: None),
     FMT_2DP),
]:
    write_formula_row(ws_cfs, next_r_cfs, lbl, fn, fmt); next_r_cfs += 1

next_r_cfs += 1
cfs_stat_lbls = [lbl for lbl in cfs_map
                 if any(k in lbl.lower() for k in ("operating activities",
                                                    "investing activities",
                                                    "financing activities",
                                                    "net change in cash"))]
next_r_cfs = write_stats_block(ws_cfs, next_r_cfs, cfs_map, cfs_stat_lbls)

# ── WACC SHEET ────────────────────────────────────────────────────────────────
ws_w = wb.create_sheet("WACC")
ws_w.column_dimensions["A"].width = 4
ws_w.column_dimensions["B"].width = 38
ws_w.column_dimensions["C"].width = 16
ws_w.column_dimensions["D"].width = 16
ws_w.column_dimensions["E"].width = 13
ws_w.column_dimensions["F"].width = 13
ws_w.column_dimensions["G"].width = 13

def w_title(r, text):
    ws_w.merge_cells(f"B{r}:G{r}")
    c = ws_w[f"B{r}"]
    c.value = text
    c.font  = Font(bold=True, size=14, color="FFFFFF", name="Calibri")
    c.fill  = PatternFill("solid", fgColor=NAVY)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws_w.row_dimensions[r].height = 26

def w_sec(r, text):
    ws_w.merge_cells(f"B{r}:G{r}")
    c = ws_w[f"B{r}"]
    c.value = text
    c.font  = Font(bold=True, size=10, color="FFFFFF", name="Calibri")
    c.fill  = PatternFill("solid", fgColor="2E5F8A")
    ws_w.row_dimensions[r].height = 16

def w_row(r, label, value=None, formula=None, fmt=FMT_PCT, is_input=False, is_result=False):
    lc = ws_w.cell(row=r, column=2, value=label)
    lc.font = Font(bold=is_result, size=10, name="Calibri")
    if is_result: lc.fill = PatternFill("solid", fgColor=WACC_RSLT)
    vc = ws_w.cell(row=r, column=3, value=formula if formula else value)
    vc.number_format = fmt
    vc.alignment = Alignment(horizontal="center")
    if is_input:
        vc.fill = PatternFill("solid", fgColor=WACC_INPUT)
        vc.font = Font(color="0000FF", size=10, name="Calibri")
    elif is_result:
        vc.fill = PatternFill("solid", fgColor=WACC_RSLT)
        vc.font = Font(bold=True, color="375623", size=11, name="Calibri")
    else:
        vc.font = Font(color="000000", size=10, name="Calibri")
    ws_w.row_dimensions[r].height = 15

w_title(2, f"{company_name}  |  WACC Buildup")
ws_w["B3"].value = "Yellow = user inputs.  Green = WACC result."
ws_w["B3"].font  = Font(italic=True, size=9, color="808080", name="Calibri")

w_sec(5, "COST OF DEBT")
w_row(6,  "Pre-tax cost of debt  (Kd)",            value=0.045, fmt=FMT_PCT, is_input=True)
w_row(7,  "Marginal tax rate  (T)",                 value=0.21,  fmt=FMT_PCT, is_input=True)
w_row(8,  "After-tax cost of debt  Kd \u00d7 (1-T)", formula="=C6*(1-C7)", fmt=FMT_PCT)

w_sec(10, "COST OF EQUITY  \u2014  CAPM")
w_row(11, "Risk-free rate  (Rf)  10-yr UST",        value=0.043,  fmt=FMT_PCT, is_input=True)
w_row(12, "Equity beta  (\u03b2)",                  value=1.20,   fmt=FMT_2DP, is_input=True)
w_row(13, "Equity risk premium  (ERP)",              value=0.055,  fmt=FMT_PCT, is_input=True)
w_row(14, "Cost of equity  Ke = Rf + \u03b2 \u00d7 ERP", formula="=C11+C12*C13", fmt=FMT_PCT)

w_sec(16, "CAPITAL STRUCTURE")
for cn, txt in [(2,"Component"),(3,"Market Value ($MM)"),(4,"Target Override"),(5,"Weight")]:
    c = ws_w.cell(row=17, column=cn, value=txt)
    c.font = Font(bold=True, size=9, name="Calibri")
    c.fill = PatternFill("solid", fgColor=MED_GRAY)
    c.alignment = Alignment(horizontal="center")
ws_w.row_dimensions[17].height = 14

for r_, lbl, we_formula in [
    (18, "Equity (Market Capitalization)",  "=IF(D18<>\"\",D18,C18/(C18+C19))"),
    (19, "Net Debt  (Total Debt \u2212 Cash)", "=1-E18"),
]:
    ws_w.cell(row=r_, column=2, value=lbl).font = Font(size=10, name="Calibri")
    inp = ws_w.cell(row=r_, column=3)
    inp.fill = PatternFill("solid", fgColor=WACC_INPUT)
    inp.font = Font(color="0000FF", size=10, name="Calibri")
    inp.number_format = FMT_DOLLAR
    inp.alignment = Alignment(horizontal="center")
    we = ws_w.cell(row=r_, column=5, value=we_formula)
    we.number_format = FMT_PCT
    we.font = Font(color="000000", size=10, name="Calibri")
    we.alignment = Alignment(horizontal="center")
    ws_w.row_dimensions[r_].height = 15

w_sec(21, "WACC")
ws_w.cell(row=22, column=2,
          value="WACC  =  Ke \u00d7 We  +  Kd(1\u2212T) \u00d7 Wd").font = Font(bold=True, size=11, name="Calibri")
ws_w.cell(row=22, column=2).fill = PatternFill("solid", fgColor=WACC_RSLT)
wr = ws_w.cell(row=22, column=3, value="=C14*E18+C8*E19")
wr.number_format = FMT_PCT
wr.font      = Font(bold=True, size=13, color="375623", name="Calibri")
wr.fill      = PatternFill("solid", fgColor=WACC_RSLT)
wr.alignment = Alignment(horizontal="center", vertical="center")
ws_w.row_dimensions[22].height = 22

w_sec(25, "SENSITIVITY  \u2014  WACC  vs  Beta  &  ERP")
ws_w.cell(row=26, column=2, value="WACC").font = Font(bold=True, size=9, name="Calibri")
ws_w.cell(row=26, column=3, value="Beta \u2192").font = Font(bold=True, size=9, name="Calibri", color="595959")
ws_w.cell(row=27, column=2, value="ERP \u2193").font  = Font(bold=True, size=9, name="Calibri", color="595959")
BETA_OFFSETS = [-0.30, -0.15, 0.00, 0.15, 0.30]
ERP_OFFSETS  = [-0.010, -0.005, 0.000, 0.005, 0.010]
for j, bo in enumerate(BETA_OFFSETS):
    beta_f = "=C12" if bo == 0 else f"={bo:+.2f}+C12"
    c = ws_w.cell(row=26, column=3+j, value=beta_f)
    c.number_format = FMT_2DP
    c.font = Font(bold=True, size=9, name="Calibri")
    c.fill = PatternFill("solid", fgColor=MED_GRAY)
    c.alignment = Alignment(horizontal="center")
for i, eo in enumerate(ERP_OFFSETS):
    erp_f = "=C13" if eo == 0 else f"={eo:+.3f}+C13"
    er = ws_w.cell(row=27+i, column=2, value=erp_f)
    er.number_format = FMT_PCT
    er.font = Font(bold=True, size=9, name="Calibri")
    er.fill = PatternFill("solid", fgColor=MED_GRAY)
    er.alignment = Alignment(horizontal="center")
    for j, bo in enumerate(BETA_OFFSETS):
        beta_expr = "C12" if bo == 0 else f"({bo:+.2f}+C12)"
        erp_expr  = "C13" if eo == 0 else f"({eo:+.3f}+C13)"
        f = f"=(C11+{beta_expr}*{erp_expr})*E18+C8*E19"
        vc = ws_w.cell(row=27+i, column=3+j, value=f)
        vc.number_format = FMT_PCT
        vc.font = Font(size=9, name="Calibri")
        vc.alignment = Alignment(horizontal="center")
        if i == 2 and j == 2:
            vc.fill = PatternFill("solid", fgColor=WACC_RSLT)
            vc.font = Font(bold=True, size=9, color="375623", name="Calibri")
    ws_w.row_dimensions[27+i].height = 14

ws_w.cell(row=34, column=2, value="Notes").font = Font(bold=True, size=9, name="Calibri")
for i, note in enumerate([
    "\u2022 Yellow cells are user inputs. Update Rf, \u03b2, ERP, market cap, and net debt.",
    "\u2022 Risk-free rate: current 10-year US Treasury yield.",
    "\u2022 ERP: Damodaran implied ERP (~5.5% for US equities, Jan 2025).",
    "\u2022 Beta: 5-year monthly vs. S&P 500 (Bloomberg / Yahoo Finance).",
    "\u2022 Cell C22 (WACC) can be referenced directly in a DCF sheet.",
], start=35):
    ws_w.cell(row=i, column=2, value=note).font = Font(size=9, name="Calibri", color="595959")
    ws_w.row_dimensions[i].height = 13

# ── AS-REPORTED LABELS SHEET ─────────────────────────────────────────────────
# Full audit trail: statement, label exactly as filed, indent level, values
ws_raw = wb.create_sheet("As-Reported Labels")
ws_raw.freeze_panes = "A2"
ws_raw.append(["Statement", "Label (as filed)", "Indent (em)"] + FY_LABELS)

raw_rows = []
for stmt, df_map, lvl_map in [
    ("Income Statement",    df_is,  is_levels),
    ("Balance Sheet",       df_bs,  bs_levels),
    ("Cash Flow Statement", df_cfs, cfs_levels),
]:
    for label in df_map.index:
        row = [stmt, label, lvl_map.get(label, 0)]
        for fy in FY_LABELS:
            v = df_map.loc[label, fy]
            row.append(float(v) if pd.notna(v) and v is not None else None)
        ws_raw.append(row)
        raw_rows.append(row)

n_raw = len(raw_rows) + 1
tbl   = Table(displayName="AsReportedLabels",
              ref=f"A1:{col(3+N_FY)}{n_raw}")
tbl.tableStyleInfo = TableStyleInfo(name="TableStyleMedium9", showRowStripes=True)
ws_raw.add_table(tbl)
ws_raw.column_dimensions["A"].width = 20
ws_raw.column_dimensions["B"].width = 55
ws_raw.column_dimensions["C"].width = 12
for i in range(N_FY):
    ws_raw.column_dimensions[col(4+i)].width = 13

print("\u2705 Excel workbook built successfully")

✅ Excel workbook built successfully


In [ ]:
# ── Cell 7: Save File & Print Summary ────────────────────────────────────────

wb.save(OUTPUT_FILE)
print(f"\u2705 Saved: {OUTPUT_FILE}")
print()
print("Summary")
print("-------")
print(f"  Company          : {company_name}")
print(f"  Ticker           : {TICKER.upper()}")
print(f"  CIK              : {CIK}")
print(f"  Fiscal Years     : {', '.join(FY_LABELS)}")
print(f"  Forecast Columns : {', '.join(PROJ_LABELS)}")
print(f"  IS rows          : {len(df_is)}")
print(f"  BS rows          : {len(df_bs)}")
print(f"  CFS rows         : {len(df_cfs)}")
print(f"  Output file      : {OUTPUT_FILE}")

✅ Saved: OKE_10K_Historical_2026-04-21.xlsx

Summary
-------
  Company          : ONEOK INC /NEW/
  Ticker           : OKE
  CIK              : 0001039684
  Fiscal Years     : FY2021, FY2022, FY2023, FY2024, FY2025
  Forecast Columns : FY2026E, FY2027E, FY2028E
  IS rows          : 57
  BS rows          : 82
  CFS rows         : 64
  Output file      : OKE_10K_Historical_2026-04-21.xlsx
